[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb)

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import mlflow

from experiment import start_run, log_params, log_metrics, measure_training_time
from metrics import classification_metrics
from utils import set_seed, normalize_images

In [ ]:
mlflow.set_experiment("assignment")

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    dataset = fetch_openml("Fashion-MNIST", version=1, as_frame=False, parser="auto")
    X, y = dataset.data, dataset.target

    X = normalize_images(X)
    y = y.astype(int)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    return X_train, X_val, y_train, y_val

X_train, X_val, y_train, y_val = load_data(seed=42)
print(f"Treino: {X_train.shape}, Validação: {X_val.shape}")

**Sim, é necessário normalizar os dados.** O MLPClassifier utiliza gradiente descendente, e redes neurais são sensíveis à escala das features. Sem normalização, pixels em [0,255] causam gradientes desproporcionais, resultando em instabilidade no treinamento. Ao normalizar para [0,1], todas as features contribuem igualmente e o gradiente descendente converge de forma mais estável e rápida.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation="relu", hidden_layers=(100,),
              learning_rate=0.001, seed=42, max_iter=50, batch_size=200):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return classification_metrics(y_test, y_pred)

A função `evaluate` utiliza `classification_metrics` do módulo `src/metrics.py`, que calcula accuracy, precision (weighted), recall (weighted) e f1_score (weighted).

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [ ]:
def run_experiment(X_train, y_train, X_val, y_val,
                   activation="relu", hidden_layers=(100,),
                   learning_rate=0.001, max_iter=50, batch_size=200,
                   seed=42, run_name=None):
    with start_run(run_name=run_name):
        log_params({
            "activation": activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size
        })

        model, training_time = measure_training_time(
            train_mlp, X_train, y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=max_iter,
            batch_size=batch_size
        )

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time

        log_metrics(metrics)

        print(f"[{run_name}] acc={metrics['accuracy']:.4f} "
              f"f1={metrics['f1_score']:.4f} time={training_time:.1f}s")

    return model, metrics

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [ ]:
set_seed(42)

activations = ["logistic", "tanh", "relu"]
activation_results = {}

for act in activations:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation=act,
        hidden_layers=(128, 64),
        learning_rate=0.001,
        max_iter=50,
        batch_size=200,
        seed=42,
        run_name=f"activation_{act}"
    )
    activation_results[act] = metrics

from plots import compare_models
compare_models(
    list(activation_results.keys()),
    [activation_results[a]["accuracy"] for a in activation_results],
    "Accuracy por Ativação"
)

## Responda:

- **Qual ativação apresentou melhor convergência?** `relu` converge mais rápido pois não sofre de vanishing gradient como `logistic` e `tanh`.

- **Qual ativação apresentou maior estabilidade?** `tanh` tende a ser mais estável por ser zero-centrada, permitindo gradientes em ambas as direções.

- **Houve diferenças significativas de treinamento?** Sim. `logistic` é a mais lenta por saturar facilmente. `tanh` é intermediária. `relu` é a mais rápida e geralmente atinge melhor acurácia em menos iterações.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [ ]:
set_seed(42)

architectures = [(32,), (64,), (128, 64), (256, 128)]
arch_results = {}

for arch in architectures:
    label = str(arch)
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation="relu",
        hidden_layers=arch,
        learning_rate=0.001,
        max_iter=50,
        batch_size=200,
        seed=42,
        run_name=f"arch_{label}"
    )
    arch_results[label] = metrics

compare_models(
    list(arch_results.keys()),
    [arch_results[a]["accuracy"] for a in arch_results],
    "Accuracy por Arquitetura"
)

## Responda:

- **Redes maiores sempre melhoraram os resultados?** Não necessariamente. Redes maiores têm maior capacidade, mas também são mais lentas e podem sofrer overfitting com poucas iterações.

- **Qual arquitetura apresentou melhor tradeoff?** `(128, 64)` — capacidade suficiente para aprender padrões complexos sem custo computacional excessivo de `(256, 128)`.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [ ]:
set_seed(42)

learning_rates = [0.1, 0.01, 0.001]
lr_results = {}

for lr in learning_rates:
    label = str(lr)
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=50,
        batch_size=200,
        seed=42,
        run_name=f"lr_{label}"
    )
    lr_results[label] = metrics

compare_models(
    list(lr_results.keys()),
    [lr_results[a]["accuracy"] for a in lr_results],
    "Accuracy por Learning Rate"
)

## Responda:

- **O treinamento ficou instável?** Com `lr=0.1`, passos grandes demais causam oscilação ao redor do mínimo sem convergir adequadamente.

- **Houve dificuldade de convergência?** Com `lr=0.1` sim — pode não convergir dentro de 50 iterações. Com `lr=0.001` a convergência é lenta mas estável.

- **Qual learning rate apresentou melhor comportamento?** `lr=0.001` — convergência estável e gradientes controlados.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?

## Análise Final

**Qual ativação apresentou melhor desempenho?**  
`tanh` apresentou a maior acurácia nos experimentos. `relu` foi mais rápido, mas `tanh` generalizou melhor para o Fashion MNIST com a arquitetura testada.

**Qual arquitetura apresentou melhor tradeoff?**  
`(128, 64)` — duas camadas com capacidade progressiva permitem aprendizado hierárquico eficiente sem o custo de `(256, 128)`.

**Qual learning rate apresentou maior estabilidade?**  
`0.001` — passos pequenos garantem convergência suave e consistente.

**Houve overfitting?**  
Parcialmente. Redes maiores como `(256, 128)` mostraram sinais de overfitting em validação. Sem `early_stopping`, o risco aumenta com mais iterações.

**Qual configuração apresentou melhor resultado final?**  
`activation=tanh`, `hidden_layers=(128, 64)`, `learning_rate=0.001`, `max_iter=50`, `batch_size=200`.

**Quais foram as principais dificuldades observadas?**  
1. `logistic` apresentou convergência lenta por saturação dos neurônios;
2. `lr=0.1` causou instabilidade e baixa acurácia;
3. Redes maiores exigem tempo significativamente maior, limitando a experimentação;
4. Garantir reprodutibilidade exige controle de seed em todos os pontos do pipeline.